In [2]:
# 06. 월간 재방문 및 재구매 분석
# 10월 사용자의 11월 재방문 여부와 10월 구매자의 11월 재구매 행동을 분석

In [4]:
import duckdb

parquet_path = r"..\data\processed\2019-*.parquet"

In [6]:
# 10월 활동 사용자 중 11월에도 활동한 사용자 수와 재방문율 계산
duckdb.sql(f"""
    WITH oct_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
    ),
    nov_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
    )
    SELECT
        COUNT(*) AS returning_users,
        (SELECT COUNT(*) FROM oct_users) AS oct_users,
        ROUND(
            COUNT(*) * 100.0 /
            (SELECT COUNT(*) FROM oct_users),
            2
        ) AS revisit_rate_pct
    FROM oct_users o
    INNER JOIN nov_users n
        ON o.user_id = n.user_id
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬───────────┬──────────────────┐
│ returning_users │ oct_users │ revisit_rate_pct │
│      int64      │   int64   │      double      │
├─────────────────┼───────────┼──────────────────┤
│         1401758 │   3022290 │            46.38 │
└─────────────────┴───────────┴──────────────────┘



In [8]:
# 10월 구매 고객 중 11월에도 다시 구매한 고객 수와 재구매율 계산
duckdb.sql(f"""
    WITH oct_buyers AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
          AND event_type = 'purchase'
    ),
    nov_buyers AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
          AND event_type = 'purchase'
    )
    SELECT
        COUNT(*) AS repeat_buyers,
        (SELECT COUNT(*) FROM oct_buyers) AS oct_buyers,
        ROUND(
            COUNT(*) * 100.0 /
            (SELECT COUNT(*) FROM oct_buyers),
            2
        ) AS repurchase_rate_pct
    FROM oct_buyers o
    INNER JOIN nov_buyers n
        ON o.user_id = n.user_id
""").show()

┌───────────────┬────────────┬─────────────────────┐
│ repeat_buyers │ oct_buyers │ repurchase_rate_pct │
│     int64     │   int64    │       double        │
├───────────────┼────────────┼─────────────────────┤
│         91286 │     347118 │                26.3 │
└───────────────┴────────────┴─────────────────────┘



In [12]:
# 확인 결과
# - 10월 활동 사용자 중 46.38%가 11월에도 다시 활동
# - 10월 구매 고객 중 26.30%가 11월에도 다시 구매
# - 재방문이 실제 재구매로 이어지는 비율은 더 낮음
# - 장기 리텐션이 아니라 10월 → 11월의 1개월 후 재방문/재구매 지표로 해석
# - 46.38%와 26.30%는 신규 고객 리텐션율이 아님, 데이터가 10월부터 시작해서 10월 이전 행동을 모르기 때문에 10월 사용자 중 기존 고객이 섞여 있을 수 있음

In [14]:
# 10월 방문 세션 수에 따라 11월 재방문율이 어떻게 달라지는지 확인
duckdb.sql(f"""
    WITH oct_behavior AS (
        SELECT
            user_id,
            COUNT(DISTINCT user_session) AS oct_sessions
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
          AND user_session IS NOT NULL
        GROUP BY user_id
    ),

    nov_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
    )

    SELECT
        CASE
            WHEN o.oct_sessions = 1 THEN '1. 1 session'
            WHEN o.oct_sessions <= 3 THEN '2. 2-3 sessions'
            WHEN o.oct_sessions <= 5 THEN '3. 4-5 sessions'
            WHEN o.oct_sessions <= 10 THEN '4. 6-10 sessions'
            ELSE '5. 11+ sessions'
        END AS session_range,

        COUNT(*) AS oct_users,

        SUM(CASE
            WHEN n.user_id IS NOT NULL THEN 1
            ELSE 0
        END) AS returning_users,

        ROUND(
            SUM(CASE WHEN n.user_id IS NOT NULL THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*),
            2
        ) AS revisit_rate_pct

    FROM oct_behavior o
    LEFT JOIN nov_users n
      ON o.user_id = n.user_id

    GROUP BY session_range
    ORDER BY session_range
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬───────────┬─────────────────┬──────────────────┐
│  session_range   │ oct_users │ returning_users │ revisit_rate_pct │
│     varchar      │   int64   │     int128      │      double      │
├──────────────────┼───────────┼─────────────────┼──────────────────┤
│ 1. 1 session     │   1475463 │          468280 │            31.74 │
│ 2. 2-3 sessions  │    859777 │          435908 │             50.7 │
│ 3. 4-5 sessions  │    296857 │          194921 │            65.66 │
│ 4. 6-10 sessions │    251193 │          187468 │            74.63 │
│ 5. 11+ sessions  │    139000 │          115181 │            82.86 │
└──────────────────┴───────────┴─────────────────┴──────────────────┘



In [16]:
# 확인 결과
# - 10월 방문 세션 수가 많을수록 11월 재방문율이 지속적으로 상승
# - 1회 방문 사용자의 재방문율은 31.74%, 11회 이상 방문 사용자는 82.86%
# - 반복 방문 빈도는 다음 달 재방문 가능성과 강하게 연결된 행동 신호로 보임
# - 단, 방문 횟수가 재방문의 원인이라고 단정할 수는 없음

In [18]:
# 10월 구매 횟수에 따라 11월 재구매율이 어떻게 달라지는지 확인
duckdb.sql(f"""
    WITH oct_behavior AS (
        SELECT
            user_id,
            COUNT(*) AS oct_purchase_events
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
          AND event_type = 'purchase'
        GROUP BY user_id
    ),

    nov_buyers AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
          AND event_type = 'purchase'
    )

    SELECT
        CASE
            WHEN o.oct_purchase_events = 1 THEN '1. 1 purchase'
            WHEN o.oct_purchase_events <= 3 THEN '2. 2-3 purchases'
            WHEN o.oct_purchase_events <= 5 THEN '3. 4-5 purchases'
            WHEN o.oct_purchase_events <= 10 THEN '4. 6-10 purchases'
            ELSE '5. 11+ purchases'
        END AS purchase_range,

        COUNT(*) AS oct_buyers,

        SUM(CASE
            WHEN n.user_id IS NOT NULL THEN 1
            ELSE 0
        END) AS repeat_buyers,

        ROUND(
            SUM(CASE WHEN n.user_id IS NOT NULL THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*),
            2
        ) AS repurchase_rate_pct

    FROM oct_behavior o
    LEFT JOIN nov_buyers n
      ON o.user_id = n.user_id

    GROUP BY purchase_range
    ORDER BY purchase_range
""").show()

┌───────────────────┬────────────┬───────────────┬─────────────────────┐
│  purchase_range   │ oct_buyers │ repeat_buyers │ repurchase_rate_pct │
│      varchar      │   int64    │    int128     │       double        │
├───────────────────┼────────────┼───────────────┼─────────────────────┤
│ 1. 1 purchase     │     215691 │         43143 │                20.0 │
│ 2. 2-3 purchases  │      91443 │         27850 │               30.46 │
│ 3. 4-5 purchases  │      20439 │          8900 │               43.54 │
│ 4. 6-10 purchases │      12938 │          6937 │               53.62 │
│ 5. 11+ purchases  │       6607 │          4456 │               67.44 │
└───────────────────┴────────────┴───────────────┴─────────────────────┘



In [20]:
# 확인 결과
# - 10월 구매 횟수가 많을수록 11월 재구매율이 지속적으로 상승
# - 1회 구매 고객의 재구매율은 20.00%, 11회 이상 구매 고객은 67.44%
# - 반복 구매 경험이 많은 고객일수록 다음 달 재구매 가능성이 높은 패턴
# - 단, 과거 구매 횟수가 재구매의 원인이라고 단정할 수는 없음

In [24]:
#--------------------------------------------------------------------------------------------------------------------------#

In [26]:
# Tableau용 10월 방문 세션 수별 11월 재방문율 데이터 생성
revisit_by_sessions = duckdb.sql(f"""
    WITH oct_behavior AS (
        SELECT
            user_id,
            COUNT(DISTINCT user_session) AS oct_sessions
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
          AND user_session IS NOT NULL
        GROUP BY user_id
    ),

    nov_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
    )

    SELECT
        CASE
            WHEN o.oct_sessions = 1 THEN '1. 1 session'
            WHEN o.oct_sessions <= 3 THEN '2. 2-3 sessions'
            WHEN o.oct_sessions <= 5 THEN '3. 4-5 sessions'
            WHEN o.oct_sessions <= 10 THEN '4. 6-10 sessions'
            ELSE '5. 11+ sessions'
        END AS session_range,

        COUNT(*) AS oct_users,

        SUM(CASE
            WHEN n.user_id IS NOT NULL THEN 1
            ELSE 0
        END) AS returning_users,

        ROUND(
            SUM(CASE WHEN n.user_id IS NOT NULL THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*),
            2
        ) AS revisit_rate_pct

    FROM oct_behavior o
    LEFT JOIN nov_users n
      ON o.user_id = n.user_id

    GROUP BY session_range
    ORDER BY session_range
""").df()

# Tableau용 CSV로 저장
revisit_by_sessions.to_csv(
    r"..\data\marts\dashboard_revisit_by_sessions.csv",
    index=False
)

revisit_by_sessions

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,session_range,oct_users,returning_users,revisit_rate_pct
0,1. 1 session,1475463,468280.0,31.74
1,2. 2-3 sessions,859777,435908.0,50.70
2,3. 4-5 sessions,296857,194921.0,65.66
3,4. 6-10 sessions,251193,187468.0,74.63
4,5. 11+ sessions,139000,115181.0,82.86


In [28]:
# Tableau용 10월 구매 횟수별 11월 재구매율 데이터 생성
repurchase_by_purchases = duckdb.sql(f"""
    WITH oct_behavior AS (
        SELECT
            user_id,
            COUNT(*) AS oct_purchase_events
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
          AND event_type = 'purchase'
        GROUP BY user_id
    ),

    nov_buyers AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
          AND event_type = 'purchase'
    )

    SELECT
        CASE
            WHEN o.oct_purchase_events = 1 THEN '1. 1 purchase'
            WHEN o.oct_purchase_events <= 3 THEN '2. 2-3 purchases'
            WHEN o.oct_purchase_events <= 5 THEN '3. 4-5 purchases'
            WHEN o.oct_purchase_events <= 10 THEN '4. 6-10 purchases'
            ELSE '5. 11+ purchases'
        END AS purchase_range,

        COUNT(*) AS oct_buyers,

        SUM(CASE
            WHEN n.user_id IS NOT NULL THEN 1
            ELSE 0
        END) AS repeat_buyers,

        ROUND(
            SUM(CASE WHEN n.user_id IS NOT NULL THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*),
            2
        ) AS repurchase_rate_pct

    FROM oct_behavior o
    LEFT JOIN nov_buyers n
      ON o.user_id = n.user_id

    GROUP BY purchase_range
    ORDER BY purchase_range
""").df()

# Tableau용 CSV로 저장
repurchase_by_purchases.to_csv(
    r"..\data\marts\dashboard_repurchase_by_purchases.csv",
    index=False
)

repurchase_by_purchases

,purchase_range,oct_buyers,repeat_buyers,repurchase_rate_pct
0,1. 1 purchase,215691,43143.0,20.00
1,2. 2-3 purchases,91443,27850.0,30.46
2,3. 4-5 purchases,20439,8900.0,43.54
3,4. 6-10 purchases,12938,6937.0,53.62
4,5. 11+ purchases,6607,4456.0,67.44
